# Análise dos resultados — DE/rand/1/bin autoadaptativo

Este notebook registra e apresenta os resultados produzidos pelas versões desenvolvidas em `work_3/truss`. A implementação corrente minimiza a função de Rastrigin bidimensional com Evolução Diferencial autoadaptativa. A análise considera execuções independentes e preserva as sementes e os orçamentos de avaliação armazenados nos arquivos CSV.

> **Escopo.** O gráfico gerado por `plot_results.py` descreve a evolução da distribuição de aptidão da população na melhor execução de cada configuração. Ele não constitui, isoladamente, uma comparação estatística entre algoritmos.

## 1. Preparação e leitura dos dados

Os resultados finais são lidos de `results/`, enquanto as séries de convergência são lidas de `evolution_curves/`. Caminhos absolutos registrados nos CSVs não são necessários para a análise, o que facilita sua reprodução em outro ambiente.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

candidate = Path.cwd()
BASE_DIR = candidate if (candidate / 'plot_results.py').exists() else candidate / 'work_3' / 'truss'
if not (BASE_DIR / 'plot_results.py').exists():
    raise FileNotFoundError('Execute o notebook a partir de sua pasta ou da raiz do repositório.')

RESULTS_DIR = BASE_DIR / 'results'
CURVES_DIR = BASE_DIR / 'evolution_curves'
PLOTS_DIR = BASE_DIR / 'evolution_plots'

result_files = sorted(RESULTS_DIR.glob('*.csv'))
if not result_files:
    raise FileNotFoundError(f'Nenhum resultado CSV encontrado em {RESULTS_DIR}')

results = pd.concat((pd.read_csv(path) for path in result_files), ignore_index=True)
print(f'{len(results)} execuções carregadas de {len(result_files)} arquivo(s).')

## 2. Síntese das execuções independentes

Para cada configuração, são reportados número de execuções, melhor valor, mediana, média, desvio-padrão amostral e pior valor de `best_f`. Como o problema é de minimização, valores menores são melhores. A mediana e a dispersão entre sementes são essenciais, pois uma única execução não caracteriza adequadamente um algoritmo estocástico.

In [ ]:
configuration_columns = [
    'algorithm', 'population_size', 'dimension',
    'max_fitness_evaluations', 'evaluation_step',
]
summary = (
    results.groupby(configuration_columns, dropna=False)['best_f']
    .agg(execucoes='count', melhor='min', mediana='median', media='mean', desvio_padrao='std', pior='max')
    .reset_index()
)
summary

In [ ]:
best_index = results.groupby(configuration_columns, dropna=False)['best_f'].idxmin()
best_runs = results.loc[best_index].sort_values(configuration_columns).reset_index(drop=True)
best_columns = configuration_columns + [
    'seed', 'best_f', 'best_x_0', 'best_x_1',
    'mean_differential_weight', 'std_differential_weight',
    'mean_crossover_rate', 'std_crossover_rate',
]
best_runs[best_columns]

## 3. Registro acadêmico breve

Nos dados atualmente armazenados, foram realizadas **20 execuções independentes** (seeds 1–20), com população de 30 indivíduos e orçamento de 3.030 avaliações. O melhor valor foi **0,0**, obtido pela seed 9 no ponto $(-2{,}5331 \times 10^{-9}, 6{,}4690 \times 10^{-10})$. A mediana de `best_f` foi $4{,}7802 \times 10^{-12}$; a média, $7{,}5630 \times 10^{-9}$; e o desvio-padrão amostral, $2{,}7216 \times 10^{-8}$. Dezoito das vinte execuções atingiram `best_f` $\leq 10^{-8}$. Na população final da melhor execução, obteve-se $F = 0{,}3337 \pm 0{,}0075$ e $CR = 0{,}9084 \pm 0{,}0081$. Esses resultados indicam aproximação numericamente elevada ao ótimo conhecido da Rastrigin 2D, mas não constituem comparação com outro algoritmo.

A célula seguinte recalcula automaticamente esse relato a partir dos CSVs presentes no momento da execução, mantendo o registro sincronizado com novos experimentos. O limiar de $10^{-8}$ é apenas um indicador numérico de proximidade ao ótimo conhecido, $f(0,0)=0$, e não um teste estatístico.

In [ ]:
for keys, group in results.groupby(configuration_columns, dropna=False):
    algorithm, population, dimension, budget, step = keys
    best = group.loc[group['best_f'].idxmin()]
    success_count = int((group['best_f'] <= 1e-8).sum())
    text = f'''
**{algorithm}.** Foram analisadas **{len(group)} execuções independentes**, com população {population}, dimensão {dimension}, orçamento de {budget} avaliações da função objetivo e registro a cada {step} avaliações. O melhor valor observado foi **{best['best_f']:.6e}** (seed {int(best['seed'])}), no ponto ({best['best_x_0']:.6e}, {best['best_x_1']:.6e}). Entre as execuções, a mediana de `best_f` foi {group['best_f'].median():.6e}, a média foi {group['best_f'].mean():.6e} e o desvio-padrão amostral foi {group['best_f'].std():.6e}. **{success_count}/{len(group)}** execuções atingiram `best_f` ≤ 10⁻⁸. Na população final da melhor execução, os parâmetros adaptativos foram F = {best['mean_differential_weight']:.4f} ± {best['std_differential_weight']:.4f} e CR = {best['mean_crossover_rate']:.4f} ± {best['std_crossover_rate']:.4f}. Esses valores descrevem indivíduos da população final, e não a variabilidade entre execuções.
'''
    display(Markdown(text))

## 4. Curva da melhor execução

A rotina original seleciona, em cada configuração, a seed com menor `best_f`. A linha representa a aptidão média da população e a faixa sombreada representa um desvio-padrão populacional acima e abaixo da média. Portanto, a figura informa simultaneamente tendência de convergência e dispersão interna da população, mas não mostra a incerteza entre execuções independentes.

In [ ]:
import importlib.util

module_spec = importlib.util.spec_from_file_location('plot_results', BASE_DIR / 'plot_results.py')
plot_results = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(plot_results)

plot_results.create_plots(CURVES_DIR, RESULTS_DIR, PLOTS_DIR)
for figure_path in sorted(PLOTS_DIR.glob('melhor_execucao--*.png')):
    display(Markdown(f'**{figure_path.name}**'))
    display(Image(filename=str(figure_path)))

## 5. Interpretação e limitações

A aproximação consistente do ótimo global deve ser avaliada pelo conjunto das execuções, e não apenas pela melhor curva. A figura permite verificar a redução da aptidão média e da heterogeneidade da população ao longo do orçamento. Comparações futuras entre versões devem usar as mesmas instâncias, orçamentos e sementes, além de reportar uma medida de tendência central, dispersão e, quando apropriado, um teste estatístico pareado.